# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vikraamkumar-ds/flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Method: Random Forest classifier** (`sklearn.ensemble.RandomForestClassifier`), predicting `is_declining` from the 8 features audited in `w03`/`w04`.

**Why it fits this lane:**
- The features audited in `w04_signal_audit` don't relate to the label in a straight line — the position quartile breakdown showed a threshold-ish effect, not a smooth linear one. A tree-based model captures that kind of non-linear, interacting pattern without needing me to hand-engineer interaction terms.
- Features sit on very different scales and shapes (heavy-tailed `imp_prev30` vs. bounded `ctr_prev30`) — trees split on raw values and don't require scaling/normalizing, so the heavy tails from `w04`'s distribution check don't need to be fixed first.
- Feature importances come for free, which directly feeds Section 4's error analysis and, later, the capstone's ranked recommendations.
- It's the same method taught in Notebook 3's demo, so the comparison in Section 3 is apples-to-apples with code I already understand and can defend.


In [ ]:
%pip -q install duckdb huggingface_hub pandas scikit-learn matplotlib

import os, getpass
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'fact_daily':     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

features = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}),
    windowed AS (
        SELECT
            f.client_hash_id, f.content_hash_id,
            SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_prev30,
            AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_prev30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()
features['ctr_prev30'] = features['clk_prev30'] / features['imp_prev30']

qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)  AS visible_queries,
           ANY_VALUE(rare_impressions_share)        AS rare_share,
           ANY_VALUE(anonymized_impressions_share)  AS anon_share,
           MAX(impressions_90d)                     AS top_query_impressions,
           SUM(impressions_90d)                     AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()
qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']

data = features.merge(qsignals, on='content_hash_id', how='left')
data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

FEATURE_COLS = ['imp_prev30', 'clk_prev30', 'pos_prev30', 'ctr_prev30',
                'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=FEATURE_COLS + ['is_declining']).reset_index(drop=True)
print(f'{len(model_data):,} rows, base rate = {model_data["is_declining"].mean():.3f}')


## 2. Split design

**Grouped by `client_hash_id`, not random, and not time-aware.**

- **Why not random row split:** rows from the same client are correlated — a client's overall traffic level, industry, and content strategy leak across its own pages. A random split would let the model see some of a client's pages in training and others in test, so it could partly succeed by learning “what this particular client's pages look like” rather than a pattern that generalizes to a *new* client. That's exactly the risk `w03_feature_leakage_check` flagged for excluding `client_hash_id` as a feature — a random split would let it back in through the split itself, even though it's not a column.
- **Why not time-aware here specifically:** every row in this feature table already comes from one fixed panel (prev30 vs last30, anchored to the same `MAX(report_date)`). There's no second, later time period in this table to hold out — the time-aware split already happened when the label was defined. `GroupShuffleSplit` on client is the honest split *for this table*; a true time-aware split would require re-pulling a rolling series of panels, which is future work noted in the capstone's limitations.
- **The real test this split answers:** *does this model work on clients it has never seen, using only content signals?* That is the only claim I'm allowed to make from this split.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

X = model_data[FEATURE_COLS]
y = model_data['is_declining']
groups = model_data['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

# confirm zero client overlap between train and test -- the whole point of this split
train_clients = set(groups.iloc[train_idx])
test_clients  = set(groups.iloc[test_idx])
overlap = train_clients & test_clients
print(f'train rows: {len(X_tr):,}   test rows: {len(X_te):,}')
print(f'train clients: {len(train_clients)}   test clients: {len(test_clients)}   overlap: {len(overlap)}')
assert len(overlap) == 0, 'LEAKAGE: same client appears in both train and test'
print('PASS -- no client appears in both train and test')


## 3. Train + compare vs my baseline

Three rows, same test split, same metric:
1. **Majority-class baseline** — always predict the more common class. The floor everything must beat.
2. **Rule-based baseline** (same shape as `w04_baseline_score`'s transparent hand-rule) — flag a page as declining if `pos_prev30` sits in the worst quartile, the flag-linked signal audited in `w04_signal_audit`.
3. **Random Forest** — the trained model from Section 1, evaluated on the exact same held-out client group.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

results = []

# 1. Majority-class baseline
majority_pred = np.full(len(y_te), y_tr.mode()[0])
results.append(('Majority-class baseline', majority_pred))

# 2. Rule-based baseline: worst position quartile -> flagged as declining
pos_q75 = X_tr['pos_prev30'].quantile(0.75)  # higher number = worse position
rule_pred = (X_te['pos_prev30'] >= pos_q75).astype(int).values
results.append(('Rule-based baseline (worst position quartile)', rule_pred))

# 3. Random Forest
model = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20,
                                random_state=42, n_jobs=-1, class_weight='balanced')
model.fit(X_tr, y_tr)
rf_pred = model.predict(X_te)
results.append(('Random Forest', rf_pred))

comparison = pd.DataFrame([{
    'method': name,
    'precision': precision_score(y_te, pred, zero_division=0),
    'recall':    recall_score(y_te, pred, zero_division=0),
    'f1':        f1_score(y_te, pred, zero_division=0),
} for name, pred in results])
print(comparison.to_string(index=False))

print('\nFull report for Random Forest:')
print(classification_report(y_te, rf_pred, digits=3))


## 4. Errors and interpretation

What matters more than the metric table above: **where the model is wrong, and what it actually leans on.** Feature importances plus a look at concrete false positives/negatives, printed below (with hashed IDs only — no client-identifying detail).


In [ ]:
from sklearn.inspection import permutation_importance

# Feature importances -- impurity-based (RF built-in). Known bias: inflates importance for
# high-cardinality / continuous columns, which several of ours are (imp_prev30, rare_share, etc.),
# so it's shown alongside permutation importance below rather than trusted alone.
impurity_importances = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
print('Impurity-based importances (RF built-in, biased toward continuous features):')
print(impurity_importances)

# Permutation importance -- shuffles one feature at a time on the held-out test set and measures
# how much the score drops. Slower, but honest: it measures what the model actually RELIES on
# at prediction time, not what it happened to split on during training.
perm = permutation_importance(model, X_te, y_te, n_repeats=20, random_state=42, scoring='f1', n_jobs=-1)
perm_importances = pd.Series(perm.importances_mean, index=FEATURE_COLS).sort_values(ascending=False)
print('\nPermutation importances (test-set, f1-based, more honest):')
print(perm_importances)

# Error analysis: false positives (predicted declining, actually stable) and
# false negatives (predicted stable, actually declining)
errors = X_te.copy()
errors['actual']    = y_te.values
errors['predicted'] = rf_pred
errors['content_hash_id'] = model_data.loc[X_te.index, 'content_hash_id'].values

false_pos = errors[(errors['actual'] == 0) & (errors['predicted'] == 1)]
false_neg = errors[(errors['actual'] == 1) & (errors['predicted'] == 0)]

print(f'\nFalse positives: {len(false_pos):,}  ({len(false_pos)/len(errors):.1%} of test set)')
print(false_pos[FEATURE_COLS].describe().loc[['mean']])

print(f'\nFalse negatives: {len(false_neg):,}  ({len(false_neg)/len(errors):.1%} of test set)')
print(false_neg[FEATURE_COLS].describe().loc[['mean']])

print('\nCompare to overall test-set feature means, for context:')
print(X_te.describe().loc[['mean']])


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.